# Introduction

This notebook generates **Fig. 7** in the manuscript from optimization results obtained for hydro-chain Full-CI Hamiltonians using the VQE and VEBE method.

Target figures to reproduce:
- Fig. 7: Comparison of optimization trajectories in a certain bond length obtained by VQE and VEBE, shown as (i) the evaluated value and (ii) the logarithm of the difference between the evaluated value and the optimization target.  

Simulation data are stored in the following directories:
- `data/npz/for_paper/optimization/vebe` and `data/npz/for_paper/optimization/vqe`: **Contains the finalized datasets used to create Fig. 7 in the manuscript**.  
  This data layout is referred to as **Pattern 1** in this notebook.
- `data/npz/worked/vebe` and `data/npz/worked/vqe`: Contains results generated by executing the data collection scripts (for reproduction purposes)  
  This data layout is referred to as **Pattern 2** in this notebook.

Produced figures are stored in `data/pdf/optimization` or `data/png/optimization`.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import re
import qulacs
import sys
from datetime import datetime
from openfermion import transforms
from openfermion.transforms import jordan_wigner
from openfermion.linalg import get_sparse_operator 
from pathlib import Path
from pyscf import fci, gto, scf, mcscf
from pyscf.tools import fcidump
from scipy.sparse.linalg import eigsh

NB_DIR = Path.cwd()
PROJECT = NB_DIR.parent

try:
    from config_local import project_root  # type: ignore
    PROJECT = Path(project_root).resolve()
    #print(f"[INFO] Using local config: PROJECT={PROJECT}")
except Exception:
    print(f"[INFO] Using public layout: PROJECT={PROJECT}")

sys.path.insert(0, str(PROJECT / "scripts"))
from module.loading_tools import (
    find_rhf_files_with_metadata_for_fci, 
    get_statenpz_paths_and_values, 
    get_txt_paths_and_values, 
    get_folder_paths_and_values
    )

# === Published / vendor environment ===
# vendor_qsh = PROJECT / "vendor" / "quantum_software_handson" / "doc" / "source" / "notebooks"
# vendor_pitbe = PROJECT / "vendor" / "PItBE"

# if vendor_qsh.exists():
#     sys.path.insert(0, str(vendor_qsh))
# if vendor_pitbe.exists():
#     sys.path.insert(0, str(vendor_pitbe))

# Optional support for the legacy layout
#  (dependencies placed outside the project directory)
# EXTERNAL_BASE = PROJECT.parent
# ext_qsh = EXTERNAL_BASE / "quantum_software_handson" / "doc" / "source" / "notebooks"
# ext_pitbe = EXTERNAL_BASE / "pitbe"
# if ext_qsh.exists():
#     sys.path.insert(0, str(ext_qsh))
# if ext_pitbe.exists():
#     sys.path.insert(0, str(EXTERNAL_BASE))
#
# from qchem_util import get_molecular_hamiltonian_from_fcidump
# from pitbe import read_general

# === Local environment (author only) ===
EXTERNAL_BASE = PROJECT.parents[1]
ext_qsh = EXTERNAL_BASE / "quantum_software_handson" / "doc" / "source" / "notebooks"
ext_pitbe2 = EXTERNAL_BASE / "pitbe_2"
if ext_qsh.exists():
    sys.path.insert(0, str(ext_qsh))
if ext_pitbe2.exists():
    sys.path.insert(0, str(EXTERNAL_BASE))
from qchem_util import get_molecular_hamiltonian_from_fcidump
from pitbe_2 import read_general

In [ ]:
# === Step 0: Generate timestamp for current execution ===
now = datetime.now()
time = now.strftime("%Y_%m_%d_%H_%M")
print('This Notebook was done at ' + time)

# H2

In [ ]:
# === Step 1: Locate input data files ===
# Pattern 1: Using `for_paper` directory
folder_path = NB_DIR / ".." / "data" / "npz" / "for_paper" / "optimization" / "vebe" / "H2"
folder_path_2 = NB_DIR / ".." / "data" / "npz" / "for_paper" / "optimization" / "vqe" / "H2"
txt_path = NB_DIR / ".." / "data" / "txt"/ "for_paper" / "optimization" / "vebe" / "H2" / "50.0"

# Retrieve folder paths and corresponding parameter values.
path_list, seed_num = get_folder_paths_and_values(folder_path)
path_list_2, seed_num_2 = get_folder_paths_and_values(folder_path_2)
txt_file, dislist_h2 = get_txt_paths_and_values(txt_path)

# Pattern 2: Using `worked` directory
# folder_path = NB_DIR / ".." / "data" / "npz" / "worked" / "vebe"/ "fci" / "optimization" / "H2"/ "2026_06_24_11_54"
# folder_path_2 = NB_DIR / ".." / "data" / "npz" / "worked" / "vqe" / "fci" / "optimization" / "H2"/ "2026_06_22_02_21"
# folder_path_3 = NB_DIR / ".." / "data" / "txt"/ "worked" / "vebe" / "optimization" / "H2" / "2026_06_24_11_54"

# Retrieve folder paths and corresponding seed numbers.
# path_list, seed_num = get_folder_paths_and_values(folder_path)
# path_list_2, seed_num_2 = get_folder_paths_and_values(folder_path_2)
# path_list_3, seed_num_3 = get_folder_paths_and_values(folder_path_3)

# Select the bond length to analyze.
# bond_length_list:
# [0.55, 0.65, 0.75, 0.85, 0.95, 1.05,
#  1.15, 1.25, 1.35, 1.45, 1.55, 1.65,
#  1.75, 1.85, 1.95, 2.05, 2.15, 2.25,
#  2.35, 2.45, 2.55]
sample_number = 1


In [ ]:
# === Step 2: Loading trajectories of optimization of H2 energy ===
trajectory_vebe_list = []
trajectory_vqe_list = []

for j in range(len(path_list)):
    # === Step 2.1: Loading results of optimization of H2 energy for each methods by seed value ===
    # VEBE
    npz_file, dislist_h2 = get_statenpz_paths_and_values(path_list[j])
    all_result = np.load('' + str(npz_file[sample_number]))
    trajectory_vebe = all_result['array1']
    trajectory_vebe_list.append(trajectory_vebe)
    # VQE
    npz_file_2, dislist_h2_2 = get_statenpz_paths_and_values(path_list_2[j])
    all_result = np.load('' + str(npz_file_2[sample_number]))
    trajectory_vqe = all_result['array1']
    trajectory_vqe_list.append(trajectory_vqe)

# === Step.2.2 Search the max iteration in the optimization among seed value ===
max_len_vebe = max(len(hist) for hist in trajectory_vebe_list)
max_len_vqe = max(len(hist) for hist in trajectory_vqe_list)

# === Step.2.3 Convert list to NumPy array. ===
# VEBE
trajectory_vebe_arr = np.full((len(path_list), 
                               int(max_len_vebe)), 
                               np.nan)
for j, hist in enumerate(trajectory_vebe_list):
    trajectory_vebe_arr[j, :len(hist)] = hist
# VQE
trajectory_vqe_arr = np.full((len(path_list), 
                              int(max_len_vqe)), 
                              np.nan)
for j, hist in enumerate(trajectory_vqe_list):
    trajectory_vqe_arr[j, :len(hist)] = hist


In [ ]:
# === Step 3: Loading optimization target and shift value in VEBE from TXT file ===
# If you are using data in the `worked` directory,
# select the TXT file corresponding to the desired seed number.
# This file contains the optimization target and shift value.
# The results shown in Fig. 7 were generated using seed 50.
#txt_file, dislist_h2 = get_txt_paths_and_values(path_list_3[50])
# Optimization target
with open(txt_file[sample_number], "r") as f:
    for line in f:
        if line.startswith("True energy:"):
            optimization_target = float(line.split(":")[1].strip())
            break
# Shift value
with open(txt_file[sample_number], "r") as f:
    for line in f:
        if line.startswith("Shift value:"):
            shift_value = float(line.split(":")[1].strip())
            break

# H4

In [ ]:
# === Step 4: Locate input data files ===
# Pattern 1: Using `for_paper` directory 
npz_path = NB_DIR / ".." / "data" / "npz" / "for_paper" / "optimization" / "vebe" / "H4"
npz_path_2 = NB_DIR / ".." / "data" / "npz" / "for_paper" / "optimization" / "vqe" / "H4"
txt_path = NB_DIR / ".." / "data" / "txt"/ "for_paper" / "optimization" / "vebe" / "H4"

# Retrieve file paths and corresponding parameter values.
npz_file, dislist_h2 = get_statenpz_paths_and_values(npz_path)
npz_file_2, dislist_h2_vqe = get_statenpz_paths_and_values(npz_path_2)
txt_file, dislist_h2 = get_txt_paths_and_values(txt_path)

# Pattern 2: Using `worked` directory
# folder_path = NB_DIR / ".." / "data" / "npz" / "worked" / "vebe" / "fci" / "optimization" / "H4" / "2026_06_23_09_01"
# folder_path_2 = NB_DIR / ".." / "data" / "npz" / "worked" / "vqe" / "fci" / "optimization" / "H4" / "2026_06_22_02_42"
# folder_path_3 = NB_DIR / ".." / "data" / "txt"/ "worked" / "vebe" / "optimization" / "H4" / "2026_06_23_09_01"

# Retrieve folder paths and corresponding seed numbers.
# path_list, seed_num = get_folder_paths_and_values(folder_path)
# path_list_2, seed_num_2 = get_folder_paths_and_values(folder_path_2)
# path_list_3, seed_num_3 = get_folder_paths_and_values(folder_path_3)

# Select the bond length to analyze.
# bond_length_list:
# [0.55, 0.65, 0.75, 0.85, 0.95, 1.05,
#  1.15, 1.25, 1.35, 1.45, 1.55, 1.65,
#  1.75, 1.85, 1.95, 2.05, 2.15, 2.25,
#  2.35, 2.45, 2.55]
sample_number = 1


In [ ]:
# === Step 5: Loading trajectories of optimization of H4 energy ===
# Pattern 1: Using `for_paper` directory
# VEBE
all_result_vebe = np.load('' + str(npz_file[sample_number]))
trajectory_vebe_h4 = all_result_vebe['array1']
# VQE
all_result_vqe = np.load('' + str(npz_file_2[sample_number]))
trajectory_vqe_h4 = all_result_vqe['array1']
# Convert list to NumPy array.
trajectory_vebe_arr_h4 = np.asarray(trajectory_vebe_h4)
trajectory_vqe_arr_h4 = np.asarray(trajectory_vqe_h4)

# Pattern 2: Using `worked` directory:
# trajectory_vebe_list_h4 = []
# trajectory_vqe_list_h4 = []

# for j in range(len(path_list)):
    # === Step.5.1: Loading results of optimization of H2 energy for each methods by seed value ===
    # VEBE
    # npz_file, dislist_h4 = get_statenpz_paths_and_values(path_list[j])
    # all_result = np.load('' + str(npz_file[sample_number]))
    # trajectory_vebe_h4 = all_result['array1']
    # trajectory_vebe_list_h4.append(trajectory_vebe_h4)
    # VQE
    # npz_file_2, dislist_h4_2 = get_statenpz_paths_and_values(path_list_2[j])
    # all_result = np.load('' + str(npz_file_2[sample_number]))
    # trajectory_vqe_h4 = all_result['array1']
    # trajectory_vqe_list_h4.append(trajectory_vqe_h4)

# === Step.5.2 Search the max iteration in the optimization among seed value ===
# max_len_vebe_h4 = max(len(hist) for hist in trajectory_vebe_list_h4)
# max_len_vqe_h4 = max(len(hist) for hist in trajectory_vqe_list_h4)

# === Step.5.3 Convert list to NumPy array. ===
# VEBE
# trajectory_vebe_arr_h4 = np.full((len(path_list), 
#                                   int(max_len_vebe_h4)), 
#                                   np.nan)
# for j, hist in enumerate(trajectory_vebe_list_h4):
#     trajectory_vebe_arr_h4[j, :len(hist)] = hist
# VQE
# trajectory_vqe_arr_h4 = np.full((len(path_list), 
#                                  int(max_len_vqe_h4)), 
#                                  np.nan)
# for j, hist in enumerate(trajectory_vqe_list_h4):
#     trajectory_vqe_arr_h4[j, :len(hist)] = hist

In [ ]:
# === Step 6: Loading optimization target and shift value in VEBE from TXT file ===
# If you are using data in the `worked` directory,
# select the TXT file corresponding to the desired seed number.
# This file contains the optimization target and shift value.
# The results shown in Fig. 7 were generated using seed 100.
# txt_file, dislist_h4 = get_txt_paths_and_values(path_list_3[0])

# Optimization target
with open(txt_file[sample_number], "r") as f:
    for line in f:
        if line.startswith("True energy:"):
            optimization_target_h4 = float(line.split(":")[1].strip())
            break
# Shift value
with open(txt_file[sample_number], "r") as f:
    for line in f:
        if line.startswith("Shift value:"):
            shift_value_h4 = float(line.split(":")[1].strip())
            break

In [ ]:
# === Step 7: Ilustrate graph of optimization trajectories of H2 and H4 (Fig. 7) ===
# === Figure layout ===
fig, axes = plt.subplots(
    2, 2,
    figsize=(9,9),
    sharex=False,
    sharey=False
)

axes_flat = axes.flat
panel_labels = ["(a)", "(b)", "(c)", "(d)"]

# === Example: upper 2x2 panels: energy curves ===
ax_h2_energy = axes[0, 0]
ax_h4_energy = axes[0, 1]
ax_h2_log = axes[1, 0]
ax_h4_log = axes[1, 1]

# === Step 13.1: Graph for optimization trajectory ===
# H2
y_vebe_all = trajectory_vebe_arr - shift_value   # shape: (100, max_len)
y_vqe_all = trajectory_vqe_arr
for j in range(y_vebe_all.shape[0]):
    ax_h2_energy.plot(np.arange(y_vqe_all[j].shape[0]), y_vqe_all[j], color="red")
    ax_h2_energy.plot(np.arange(y_vebe_all[j].shape[0]), y_vebe_all[j], color="blue")
ax_h2_energy.axhline(optimization_target, color="black", linestyle="--")
ax_h2_energy.set_title("H$_2$", fontsize=13)

# H4
y_vebe = trajectory_vebe_arr_h4 - shift_value_h4
y_vqe = trajectory_vqe_arr_h4
# If you are using data in the `worked` directory,
# adjust the array shape if necessary.
# y_vebe = np.asarray(trajectory_vebe_arr_h4 - shift_value_h4).squeeze()
# y_vqe = np.asarray(trajectory_vqe_arr_h4).squeeze()
ax_h4_energy.plot(np.arange(y_vqe.shape[0]), y_vqe, color="red")
ax_h4_energy.plot(np.arange(y_vebe.shape[0]), y_vebe, color="blue")
ax_h4_energy.axhline(
    optimization_target_h4, color="black", linestyle="--"
    )
ax_h4_energy.set_title("H$_4$", fontsize=13)

# H2 log error
y_vebe_log_all = np.log10(np.abs(trajectory_vebe_arr - shift_value - optimization_target))
y_vqe_log_all = np.log10(np.abs(trajectory_vqe_arr - optimization_target))
for j in range(y_vebe_log_all.shape[0]):
    ax_h2_log.plot(np.arange(y_vqe_log_all[j].shape[0]), y_vqe_log_all[j], color="red")
    ax_h2_log.plot(np.arange(y_vebe_log_all[j].shape[0]), y_vebe_log_all[j], color="blue")
ax_h2_log.axhline(np.log10(1.6e-3), color="orange", linestyle=":")
ax_h2_log.set_title("H$_2$ log error", fontsize=13)

# H4 log error
eps = 1e-16
y_vebe_log = np.log10(
    np.abs(trajectory_vebe_arr_h4 - shift_value_h4 - optimization_target_h4)
)
y_vqe_log = np.log10(
    np.abs(trajectory_vqe_arr_h4 - optimization_target_h4)
)
# If you are using data in the `worked` directory,
# adjust the array shape if necessary.
# y_vebe_log = np.log10(
#     np.maximum(
#         np.abs(np.asarray(trajectory_vebe_arr_h4 - shift_value_h4 - optimization_target_h4).squeeze()),
#         eps,
#     )
# )
# y_vqe_log = np.log10(
#     np.maximum(
#         np.abs(np.asarray(trajectory_vqe_arr_h4 - optimization_target_h4).squeeze()),
#         eps,
#     )
# )
ax_h4_log.plot(np.arange(y_vqe_log.shape[0]), y_vqe_log, color="red")
ax_h4_log.plot(np.arange(y_vebe_log.shape[0]), y_vebe_log, color="blue")
ax_h4_log.axhline(np.log10(1.6e-3), color="orange", linestyle=":")
ax_h4_log.set_title("H$_4$ log error", fontsize=13)

# === Step 13.2: Setting some conditions ===
# Panel labels
for ax, label in zip(axes_flat, panel_labels):
    ax.text(
        -0.13,
        1.08,
        label,
        transform=ax.transAxes,
        fontsize=13,
        va="top"
    )

# Common axis labels
fig.supxlabel(r"$x$: Iteration", fontsize=13, y=0.065)

fig.text(
    0.062, 0.72,
    "Evaluated value (Hartree)",
    rotation=90,
    va="center",
    fontsize=13
)

fig.text(
    0.062, 0.28,
    r"Evaluated value ($\log$(Hartree))",
    rotation=90,
    va="center",
    fontsize=13
)

# Common x ticks
for ax in axes_flat:
    ax.tick_params(axis="both", labelsize=11)

fig.suptitle(r"Optimization process for H$_2$ and H$_4$", fontsize=16, x=0.55, y=0.92)

# Legends 
handles = [
    plt.Line2D([], [], color='red', label='VQE'),
    plt.Line2D([], [], color='blue', label='VEBE'),
    plt.Line2D([], [], color='black', linestyle='--', label='target of optimization'),
    plt.Line2D([], [], color='orange', linestyle=':', label='chemical precision')
]

fig.legend(
    handles=handles, loc='lower center',
    ncol=2, frameon=False, fontsize=13,
    bbox_to_anchor=(0.55, -0.015),
)

# Layout
#plt.tight_layout(rect=[0.05, 0.03, 1.0, 0.97])
fig.subplots_adjust(
    left=0.155,
    right=0.98,
    bottom=0.14,
    top=0.86,
    wspace=0.25,
    hspace=0.35,
)

# === Step 13.3: Plot and save figures ===
base_dir = os.path.dirname(__file__) if "__file__" in globals() else os.getcwd()
# Select the file type (ex. .pdf, .png) and its corresponding folder.
out_dir = os.path.join(
    base_dir, "..", "data", "pdf", "optimization"
)
os.makedirs(out_dir, exist_ok=True)
pdf_path = os.path.join(out_dir, f"optimization_{time}.pdf")
#plt.savefig(pdf_path, bbox_inches='tight')

plt.show()